# Manifold Sampling Demo

Sample points uniformly from compact orientable manifolds based on the principle:

$$
 \int_{\phi(U)} dV = \int_U  f(\phi(x)) \det(\phi'(x)^T \phi'(x))^\frac{1}{2}\,dx_1\dots dx_k
$$

In this document you can see two examples.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from saman import Manifold

# Sampling from an ellipse


In [ ]:
class Ellipse(Manifold):
    def __init__(self):
        super().__init__(2, 1, 2.5, np.array([0.0]), np.array([2.0 * np.pi]))

    def coord(self, t):
        return np.array([2*np.cos(t[0]), np.sin(t[0])])

    def pushforward(self, t):
        return np.array([[-2*np.sin(t[0])], [np.cos(t[0])]])

ellipse = Ellipse()

In [ ]:
samples = np.array(ellipse.sample(n_samples=35))

t = np.linspace(0, 2*np.pi, 1000)
curve = np.array([ellipse.coord(np.array([t_val])) for t_val in t])

plt.figure(figsize=(8, 5))
plt.plot(curve[:, 0], curve[:, 1], 'b-', linewidth=2, alpha=0.6)
plt.scatter(samples[:, 0], samples[:, 1], color='red', s=40, alpha=0.8)
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.title('Uniform Sampling on Ellipse')
plt.show()

# Sampling from a sphere

In [ ]:
class Sphere(Manifold):
    def __init__(self, R=1.0):
        super().__init__(3, 2, 2.0 * R * R, np.array([0.0, 0.0]), np.array([np.pi, 2.0 * np.pi]))
        self.R = R

    def coord(self, p):
        theta, phi = p
        x = self.R * np.sin(theta) * np.cos(phi)
        y = self.R * np.sin(theta) * np.sin(phi)
        z = self.R * np.cos(theta)
        return np.array([x, y, z])

    def pushforward(self, p):
        theta, phi = p
        dtheta = np.array([
            self.R * np.cos(theta) * np.cos(phi),
            self.R * np.cos(theta) * np.sin(phi),
            -self.R * np.sin(theta)
        ])
        dphi = np.array([
            -self.R * np.sin(theta) * np.sin(phi),
            self.R * np.sin(theta) * np.cos(phi),
            0.0
        ])
        return np.column_stack((dtheta, dphi))

sphere = Sphere()

In [ ]:
samples = np.array(sphere.sample(n_samples=500))

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(samples[:, 0], samples[:, 1], samples[:, 2], c="tab:blue", s=10, alpha=0.7)
ax.set_box_aspect([1, 1, 1])
plt.show()

# Sampling from a torus


In [ ]:
R, r = 1.0, 0.3

class Torus(Manifold):
    def __init__(self):
        super().__init__(3, 2, 1.5, np.array([0.0, 0.0]), np.array([2*np.pi, 2*np.pi]))

    def coord(self, p):
        u, v = p[0], p[1]
        x = (R + r * np.cos(v)) * np.cos(u)
        y = (R + r * np.cos(v)) * np.sin(u)
        z = r * np.sin(v)
        return np.array([x, y, z])
    
    def pushforward(self, p):
        u, v = p[0], p[1]
        return np.array([
            [-(R + r * np.cos(v)) * np.sin(u), -r * np.sin(v) * np.cos(u)],
            [(R + r * np.cos(v)) * np.cos(u),  -r * np.sin(v) * np.sin(u)],
            [0, r * np.cos(v)]
        ])

torus = Torus()

In [ ]:
samples = np.array(torus.sample(n_samples=200))

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Create mesh for torus surface
u_mesh = np.linspace(0, 2*np.pi, 50)
v_mesh = np.linspace(0, 2*np.pi, 50)
U, V = np.meshgrid(u_mesh, v_mesh)

X = (R + r * np.cos(V)) * np.cos(U)
Y = (R + r * np.cos(V)) * np.sin(U)
Z = r * np.sin(V)

ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.4, linewidth=0)
ax.scatter(samples[:, 0], samples[:, 1], samples[:, 2], c='red', s=30)

max_range = R + r
ax.set_xlim([-max_range, max_range])
ax.set_ylim([-max_range, max_range])
ax.set_zlim([-max_range, max_range])
ax.set_box_aspect([1, 1, 1])
plt.show()

# Mean Hausdorff distance between compact manifolds.

The mean Hausdorff distance between manifolds $M$ and $N$ is defined to be
$$
d_H(M, N)
= \frac{1}{\operatorname{vol} M} \int_M d(x, N)\,dV_M
+ \frac{1}{\operatorname{vol} N} \int_N d(x, M)\,dV_N
$$
where $d(x, N) = \min_{y\in N} |x - y|$ is the shortest distance from $x$ to the manifold $M$.
Computing this distance involves (a) finding $d(x, N) = \min_{y\in M} |x - y|$, i.e., the shortest distance
from a fixed $x\in M$ to $N$, and (b) evaluating the integrals above.

The first part can be estimated by
$$
d(x, N) = \sqrt{\min_{u \in U} |x - \phi(u)|^2}
$$
where $\phi: U \to N$ is an almost-everywhere parameterization for $N$. The second part, i.e., the integral,
can be estimated using Monte Carlo methods and manifold sampling. Let $X_1, \dots, X_n$ be a uniform sample
from the manifold $M$. This means that $d(X_1, N),\dots,d(X_n,N)$ is a sequence of iid real-valued
random variables. The law of large numbers states that
$$
\frac{1}{n}\sum_{i=1}^n d(X_i, N) \xrightarrow{\text{a.s.}} \frac{1}{\operatorname{vol}M}\int_M {d(x, N)}\,dV_M
$$
as $n\to\infty$. We will use the partial averages as estimates of this integral.

## Finding the minimum distance

As mentioned before this involves computing $\min_{u \in U} |x - \phi(u)|^2$. I will do this using scipy.

In [ ]:
def dist(x, M):
    x = np.asarray(x, dtype=float)
    loss = lambda u: np.sum((x - M.coord(u))**2)
    u0 = (M.lower + M.upper) / 2
    bounds = [(low, up) for low, up in zip(M.lower, M.upper)]
    res = minimize(loss, u0, bounds=bounds, method='L-BFGS-B')
    return np.sqrt(res.fun)

In the case of the torus, the minimum distance to the origin is $R - r = 0.7$. We can check that our function works fine

In [22]:
dist(np.zeros(torus.n), torus)

np.float64(0.7)

## Monte Carlo Integration

Lets now use our manifold sampling service to perform Monte Carlo integration.

In [ ]:
def mean_hausdorff_distance(M, N, n_samples=100):
    M_samples = M.sample(n_samples=n_samples)
    N_samples = N.sample(n_samples=n_samples)
    I = sum(dist(x, N) for x in M_samples) / n_samples
    J = sum(dist(x, M) for x in N_samples) / n_samples
    return I + J

As an example, lets find the mean Hausdorff distance between the torus and the sphere.

In [23]:
mean_hausdorff_distance(torus, sphere, n_samples=1000)

np.float64(0.5859098696278856)